In [43]:
# Cell 1 - setup

import json
import re
import numpy as np
import ollama

MODEL = "gemma4:31b-cloud"              # must be a tool-capable model
EMBED_MODEL = "embeddinggemma:latest"  # run: ollama pull nomic-embed-text

## Real tools

`search_products` is the "glue" tool - almost every other product tool needs
an ID, and this is what turns a name into one. Every tool that needs an ID
validates it and returns a helpful error if it looks wrong, instead of
silently proceeding with garbage.

In [44]:
from typing import Literal


def search_products(
    query: str,
    category: Literal["laptops", "phones", "audio", "accessories", "monitors", "all"] = "all",
    max_results: int = 5,
):
    """Search the MegaMart catalog and return matching products with price, rating, category, and product_id.

    Call this first for any question that names a product but not its ID:
    price or cost, rating, or which category an item is in. Also call it to
    obtain a product_id before using a tool that needs one (details, reviews,
    inventory, supplier).

    Args:
        query: Search keywords, e.g. 'wireless mouse' or 'AeroBook'.
        category: Restrict the search to one category, or 'all' for no filter.
        max_results: How many products to return, from 1 to 20.

    Returns:
        A list of products. Each has product_id (str), name (str),
        category (str), price_inr (int, the selling price), rating (float),
        in_catalog_since (ISO date str).
    """
    return [
        {"product_id": "P-1001", "name": "AeroBook 14", "category": "laptops",
         "price_inr": 94990, "rating": 4.4, "in_catalog_since": "2024-08-01"},
        {"product_id": "P-1002", "name": "AeroBook 16 Pro", "category": "laptops",
         "price_inr": 164990, "rating": 4.6, "in_catalog_since": "2024-11-15"},
    ][:max_results]


def get_product_details(product_id: str):
    """Get the full hardware specification sheet for one product.

    Call this after search_products has given you a product_id and the user
    asks about specs or hardware.

    Args:
        product_id: A MegaMart product ID from search_products, e.g. 'P-1001'.

    Returns:
        A dict with product_id, cpu, ram, storage, battery, weight_kg.
    """
    error = require_product_id(product_id)
    if error:
        return error
    return {"product_id": product_id, "cpu": "8-core 3.2GHz", "ram": "16GB DDR5",
            "storage": "512GB NVMe", "battery": "72Wh, 18h video", "weight_kg": 1.24}


def get_product_reviews(product_id: str):
    """Get customer reviews for a product: each reviewer's star rating and review text.

    Call this when the user asks what customers think — opinions,
    satisfaction, praise, or complaints — about a specific product.

    Args:
        product_id: A MegaMart product ID from search_products, e.g. 'P-1001'.

    Returns:
        A list of reviews, each with reviewer (str), rating (int, 1-5),
        full_text (str).
    """
    error = f"require_product_id{product_id}"
    if error:
        return error
    return [
        {"reviewer": "Priya S", "rating": 5, "full_text": "Excellent display and battery life."},
        {"reviewer": "Karan B", "rating": 2, "full_text": "Speakers crackle at high volume."},
        {"reviewer": "Sneha P", "rating": 3, "full_text": "Battery closer to 10 hours than advertised."},
    ]


def get_inventory_level(
    product_id: str,
    warehouse: Literal["delhi", "mumbai", "bangalore", "all"] = "all",
):
    """Get near-real-time stock for a product, broken down by warehouse.

    Call this for any availability or 'is it in stock' question. Numbers
    refresh about every 15 minutes. units_reserved_for_orders are already
    committed to other orders, so sellable stock is total minus reserved.

    Args:
        product_id: The MegaMart product ID.
        warehouse: Limit the count to one warehouse, or 'all' to sum them.

    Returns:
        A dict with product_id, total_units (int), by_warehouse (dict of
        warehouse name -> units), units_reserved_for_orders (int),
        restock_expected (ISO date str or null).
    """
    return {
        "product_id": product_id, "total_units": 47,
        "by_warehouse": {"delhi": 12, "mumbai": 30, "bangalore": 5},
        "units_reserved_for_orders": 8, "restock_expected": None,
    }


def get_low_stock_items(
    category: Literal["laptops", "phones", "audio", "accessories", "monitors", "all"] = "all",
    threshold: int = 10,
):
    """List products whose total stock has dropped below a threshold, with how fast they sell.

    Call this for restocking decisions and to warn staff before promising
    delivery dates on scarce items.

    Args:
        category: Restrict to one category, or 'all'.
        threshold: Count an item as low when total stock is below this value.
            Defaults to 10.

    Returns:
        A list of items, each with product_id, name, total_units (int),
        daily_sales_rate (float), days_until_stockout (int).
    """
    return [
        {"product_id": "P-1003", "name": "Vertex 14", "total_units": 2,
         "daily_sales_rate": 1.5, "days_until_stockout": 1},
        {"product_id": "P-2044", "name": "SoundCore Buds", "total_units": 7,
         "daily_sales_rate": 3.2, "days_until_stockout": 2},
    ]


def get_supplier_info(product_id: str):
    """Get a product's supplier name and restock lead time.

    Call this when planning a restock, or when a customer asks how long an
    out-of-stock item will take to come back.

    Args:
        product_id: The MegaMart product ID.

    Returns:
        A dict with supplier_name (str), lead_time_days (int),
        minimum_order_units (int), last_order_date (str).
    """
    return {
        "supplier_name": "Aero Distribution India", "lead_time_days": 9,
        "minimum_order_units": 20, "last_order_date": "2026-06-28",
    }


def create_restock_order(product_id: str, units: int):
    """Place a restock order with a product's supplier.

    Only call this after checking get_inventory_level and get_supplier_info.
    `units` must be at least the supplier's minimum_order_units or the order
    is rejected. This commits a purchase, so confirm the quantity first.

    Args:
        product_id: The MegaMart product ID.
        units: How many units to order; must meet the supplier minimum.

    Returns:
        A dict with restock_order_id (str), status (str),
        expected_arrival (YYYY-MM-DD str).
    """
    return {"restock_order_id": "RS-7781", "status": "placed",
            "expected_arrival": "2026-07-31"}


def get_order_details(order_id: str):
    """Get one order's line items, total, and status.

    Call this whenever the user's question mentions an order number.

    Args:
        order_id: The order ID, format 'ORD-' followed by digits, e.g. 'ORD-90112'.

    Returns:
        A dict with order_id, items, total_inr (int), status (str).
    """
    return {"order_id": order_id, "total_inr": 94990, "status": "delivered"}


def get_shipping_status(order_id: str):
    """Get live shipping status and the tracking history for an order.

    Call this for any 'where is my order' question. The events list is
    ordered newest first.

    Args:
        order_id: The order ID, e.g. 'ORD-90112'.

    Returns:
        A dict with order_id, courier (str), tracking_id (str),
        current_status (str), expected_delivery (YYYY-MM-DD str), and
        events (list of {timestamp, description}, newest first).
    """
    return {
        "order_id": order_id, "courier": "BlueDart", "tracking_id": "BD-4471822",
        "current_status": "out for delivery",
        "expected_delivery": "2026-07-22",
        "events": [
            {"timestamp": "2026-07-22 08:10", "description": "Out for delivery, Gurugram hub"},
            {"timestamp": "2026-07-21 19:44", "description": "Arrived at Gurugram hub"},
        ],
    }


def get_payment_status(order_id: str):
    """Check an order's payment status and transaction details.

    Call this when a customer says they were charged but the order shows
    unpaid, or when a refund has not arrived. Records update within about
    two minutes of the bank confirmation.

    Args:
        order_id: The order ID.

    Returns:
        A dict with order_id, amount_inr (int), method (str),
        status ('paid' | 'pending' | 'failed' | 'refunded'),
        transaction_id (str), paid_at (timestamp str or null).
    """
    return {"order_id": order_id, "amount_inr": 94990, "method": "UPI",
            "status": "paid", "transaction_id": "TXN-88123471",
            "paid_at": "2026-07-10 14:22"}


def process_refund(
    order_id: str,
    reason: Literal["defective", "wrong_item", "not_as_described", "changed_mind"],
):
    """Initiate a refund for a delivered or returned order.

    Only call this after confirming the order status is 'delivered' or
    'returned' AND the request is inside the 10-day return window. Refunds go
    back to the original payment method within 5-7 working days. This moves
    money, so verify the order qualifies before calling.

    Args:
        order_id: The order ID to refund.
        reason: Why the refund is being issued.

    Returns:
        A dict with refund_id (str), order_id, amount_inr (int),
        status (str), expected_completion (YYYY-MM-DD str).
    """
    return {"refund_id": "RF-3320", "order_id": order_id,
            "amount_inr": 94990, "status": "initiated",
            "expected_completion": "2026-07-29"}


def get_return_requests(
    status: Literal["pending", "approved", "rejected", "completed", "all"] = "pending",
):
    """List product return requests, filtered by their status.

    Call this to see pending returns, or to check whether an order or
    customer already has a return in progress before starting a new one.

    Args:
        status: Which returns to list, or 'all' for every status.

    Returns:
        A list of requests, each with return_id, order_id, customer_id,
        product_id, reason (str), requested_on (str), status (str).
    """
    return [
        {"return_id": "RT-1188", "order_id": "ORD-90112",
         "customer_id": "C-5501", "product_id": "P-1001",
         "reason": "defective", "requested_on": "2026-07-18",
         "status": "pending"},
    ]


def get_customer_profile(customer_id: str):
    """Get a customer's profile and account standing.

    Call this to check membership tier, contact details, and any account
    flags before making an exception or goodwill gesture for them.

    Args:
        customer_id: The customer ID, e.g. 'C-5501'.

    Returns:
        A dict with customer_id, name, email, city, member_since,
        tier ('standard' | 'gold' | 'platinum'), lifetime_spend_inr (int),
        account_flags (list of str).
    """
    return {
        "customer_id": customer_id, "name": "Rohit Verma",
        "email": "rohit.v@example.com", "city": "Gurugram",
        "member_since": "2023-02-14", "tier": "gold",
        "lifetime_spend_inr": 287400, "account_flags": [],
    }


def create_support_ticket(
    customer_id: str,
    subject: str,
    priority: Literal["low", "normal", "high"] = "normal",
):
    """Open a support ticket for an issue that cannot be resolved in chat.

    Call this when the issue needs a human specialist: hardware inspection,
    billing disputes, or physical logistics beyond a standard return.

    Args:
        customer_id: The customer ID.
        subject: One-line summary of the issue, under 100 characters.
        priority: Ticket priority. Use 'high' only for payment failures or
            safety issues.

    Returns:
        A dict with ticket_id (str), status (str),
        expected_first_response (str, e.g. 'within 4 hours').
    """
    return {"ticket_id": "TK-20991", "status": "open",
            "expected_first_response": "within 4 hours"}


def get_sales_summary(
    period: Literal["today", "last_7_days", "last_30_days", "this_quarter"] = "last_7_days",
    category: Literal["laptops", "phones", "audio", "accessories", "monitors", "all"] = "all",
):
    """Get aggregated sales figures — revenue, units, average order value, and top sellers — for a period.

    Call this for business questions about revenue or units sold. Figures
    exclude cancelled orders and include GST.

    Args:
        period: The reporting window.
        category: Restrict figures to one category, or 'all'.

    Returns:
        A dict with period, total_revenue_inr (int), units_sold (int),
        average_order_value_inr (int), top_products (list of
        {product_id, name, units}).
    """
    return {
        "period": period, "total_revenue_inr": 4823000, "units_sold": 214,
        "average_order_value_inr": 22537,
        "top_products": [
            {"product_id": "P-1001", "name": "AeroBook 14", "units": 31},
            {"product_id": "P-2044", "name": "SoundCore Buds", "units": 58},
        ],
    }

In [45]:
REAL_TOOLS = [
    search_products,
    get_product_details,
    get_product_reviews,
    get_inventory_level,
    get_low_stock_items,
    get_supplier_info,
    create_restock_order,
    get_order_details,
    get_shipping_status,
    get_payment_status,
    process_refund,
    get_return_requests,
    get_customer_profile,
    create_support_ticket,
    get_sales_summary,
]
print(len(REAL_TOOLS), "real tools defined")

15 real tools defined


In [46]:
REGISTRY = {f.__name__: f for f in REAL_TOOLS}

print(F"{len(REGISTRY)} total tools")

15 total tools


In [47]:
REGISTRY

{'search_products': <function __main__.search_products(query: str, category: Literal['laptops', 'phones', 'audio', 'accessories', 'monitors', 'all'] = 'all', max_results: int = 5)>,
 'get_product_details': <function __main__.get_product_details(product_id: str)>,
 'get_product_reviews': <function __main__.get_product_reviews(product_id: str)>,
 'get_inventory_level': <function __main__.get_inventory_level(product_id: str, warehouse: Literal['delhi', 'mumbai', 'bangalore', 'all'] = 'all')>,
 'get_low_stock_items': <function __main__.get_low_stock_items(category: Literal['laptops', 'phones', 'audio', 'accessories', 'monitors', 'all'] = 'all', threshold: int = 10)>,
 'get_supplier_info': <function __main__.get_supplier_info(product_id: str)>,
 'create_restock_order': <function __main__.create_restock_order(product_id: str, units: int)>,
 'get_order_details': <function __main__.get_order_details(order_id: str)>,
 'get_shipping_status': <function __main__.get_shipping_status(order_id: str)>

In [48]:
# Cell 10 - THE ONLY SYSTEM_PROMPT definition in this notebook.
# Generalized (not tied to product IDs specifically) and teaches chaining.

SYSTEM_PROMPT = """You are an operations assistant with access to many tools. \
search_products is already loaded and available - use it immediately whenever \
you have a product name but need its product_id. All other tools are \
discovered on demand through search_tools.

You can call search_tools as many times as needed within a single task - once \
for each distinct capability you're missing, not just once per task.

If you're missing something a tool needs - an ID, a lookup value, a prior \
result, or any other piece of data - that almost always means you're missing \
a TOOL, not missing information from the user. Search for a tool that can \
produce it. Chain tools together: use the output of one tool as the input \
to the next.

Only ask the user a question when no reasonable tool could resolve it. Never \
ask the user for something a tool could look up for you."""

In [49]:
# QUERY 1

messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": "HI,how are you"},
    ]

response = ollama.chat(
            model=MODEL,
            messages=messages,
            tools=REAL_TOOLS
        )

In [50]:
print(response)
print("*********"*10)
print(response.message.content)
print("*********"*10)
print(response.prompt_eval_count)

model='gemma4:31b' created_at='2026-07-29T05:27:12.211336771Z' done=True done_reason='stop' total_duration=597897126 load_duration=None prompt_eval_count=2071 prompt_eval_duration=None eval_count=19 eval_duration=None message=Message(role='assistant', content="I'm doing well, thank you for asking! How can I help you today?", thinking=None, images=None, tool_name=None, tool_calls=None)
******************************************************************************************
I'm doing well, thank you for asking! How can I help you today?
******************************************************************************************
2071


In [51]:
# QUERY 2 

messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": "What is price of AeroBook 14"},
    ]

response = ollama.chat(
            model=MODEL,
            messages=messages,
            tools=REAL_TOOLS
        )

In [52]:
print(response)
print("*********"*10)
print(response.message.content)
print("*********"*10)
print(response.prompt_eval_count)

model='gemma4:31b' created_at='2026-07-29T05:28:52.011254443Z' done=True done_reason='stop' total_duration=559830633 load_duration=None prompt_eval_count=2075 prompt_eval_duration=None eval_count=32 eval_duration=None message=Message(role='assistant', content='', thinking=None, images=None, tool_name=None, tool_calls=[ToolCall(function=Function(name='search_products', arguments={'category': 'all', 'max_results': 5, 'query': 'AeroBook 14'}))])
******************************************************************************************

******************************************************************************************
2075


In [54]:
# QUERY 3


messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": "Give me the sales summary for the last 7 days."},
    ]

response = ollama.chat(
            model=MODEL,
            messages=messages,
            tools=REAL_TOOLS
        )

In [55]:
print(response)
print("*********"*10)
print(response.message.content)
print("*********"*10)
print(response.prompt_eval_count)

model='gemma4:31b' created_at='2026-07-29T05:29:45.713054698Z' done=True done_reason='stop' total_duration=3391854380 load_duration=None prompt_eval_count=2078 prompt_eval_duration=None eval_count=26 eval_duration=None message=Message(role='assistant', content='', thinking=None, images=None, tool_name=None, tool_calls=[ToolCall(function=Function(name='get_sales_summary', arguments={'category': 'all', 'period': 'last 7 days'}))])
******************************************************************************************

******************************************************************************************
2078


## DEFERRED LOADING

In [56]:
# This function is a very simple tool search function. Its job is to look through all available tools and return the names of the tools 
# that seem relevant to the user's query.


def search_tools(query, max_results=3):
    words = query.lower().split()
    hits = []
    for f in REAL_TOOLS:
        text = (f.__name__ + " " + (f.__doc__ or "")).lower()  # checking for word in function name or docstring
        if any(w in text for w in words):
            hits.append(f.__name__)
    return hits[:max_results]

In [57]:
SEARCH_TOOL = {
    "type": "function",
    "function": {
        "name": "search_tools",
        "description": (
            "Search for tools that are not yet loaded into this conversation. "
            "Right now, this search tool is the only capability you have - "
            "every other tool must be found through this one before it can "
            "be called. Use it the moment you realize you need a capability "
            "you don't currently have.\n\n"
            "Returns a list of matching tool names. Once a tool is returned "
            "here, it becomes directly callable on your next turn - you do "
            "not need to search for it again later in this conversation.\n\n"
            "If nothing matches, try again with broader or different keywords "
            "rather than giving up."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": (
                        "Short keywords describing the capability you need - "
                        "not a full sentence. "
                        "Good: 'check inventory stock', 'process refund', "
                        "'sales report'. "
                        "Avoid: a complete question or a full description "
                        "of the task."
                    ),
                },
                "max_results": {
                    "type": "integer",
                    "description": (
                        "Maximum number of matching tools to return. "
                        "Defaults to 3. Increase only if the task clearly "
                        "needs several related tools at once."
                    ),
                },
            },
            "required": ["query"],
        },
    },
}

In [58]:
messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": "Hi how are you ??"},
    ]

response = ollama.chat(
            model=MODEL,
            messages=messages,
            tools=[SEARCH_TOOL]
        )

In [59]:
print(response)
print("*********"*10)
print(response.message.content)
print("*********"*10)
print(response.prompt_eval_count)

model='gemma4:31b' created_at='2026-07-29T05:52:38.671357184Z' done=True done_reason='stop' total_duration=342516425 load_duration=None prompt_eval_count=443 prompt_eval_duration=None eval_count=19 eval_duration=None message=Message(role='assistant', content="I'm doing great, thank you for asking! How can I help you today?", thinking=None, images=None, tool_name=None, tool_calls=None)
******************************************************************************************
I'm doing great, thank you for asking! How can I help you today?
******************************************************************************************
443


In [63]:
messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": "Give me the sales summary for the last 7 days."},
    ]

response = ollama.chat(
            model=MODEL,
            messages=messages,
            tools=[SEARCH_TOOL]
        )

In [64]:
print(response.prompt_eval_count)

450


In [65]:
response

ChatResponse(model='gemma4:31b', created_at='2026-07-29T05:54:20.202940225Z', done=True, done_reason='stop', total_duration=982523593, load_duration=None, prompt_eval_count=450, prompt_eval_duration=None, eval_count=17, eval_duration=None, message=Message(role='assistant', content='', thinking=None, images=None, tool_name=None, tool_calls=[ToolCall(function=Function(name='search_tools', arguments={'query': 'sales summary report'}))]))

In [66]:
response_query= search_tools(query="sales summary report")

In [67]:
response_query

['get_low_stock_items', 'create_support_ticket', 'get_sales_summary']

In [68]:
messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": "Give me the sales summary for the last 7 days."},
    ]

response = ollama.chat(
            model=MODEL,
            messages=messages,
            tools=[get_sales_summary,get_low_stock_items,create_support_ticket]

        )

In [69]:
response

ChatResponse(model='gemma4:31b', created_at='2026-07-29T05:57:06.099367547Z', done=True, done_reason='stop', total_duration=475261440, load_duration=None, prompt_eval_count=613, prompt_eval_duration=None, eval_count=26, eval_duration=None, message=Message(role='assistant', content='', thinking=None, images=None, tool_name=None, tool_calls=[ToolCall(function=Function(name='get_sales_summary', arguments={'category': 'all', 'period': 'last 7 days'}))]))

In [70]:
print(response.prompt_eval_count)

613


In [71]:
#AGENTIC LOOP FOR DEFERRED LOADING

def run_with_deferred_loading(question, max_turns=8, verbose=True):
    tools = [SEARCH_TOOL]
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]

    for turn in range(1, max_turns + 1):
        response = ollama.chat(
            model=MODEL,
            messages=messages,
            tools=tools,
            options={"num_ctx":8000},   # FIX #1 - avoid silent truncation
        )

        if verbose:
            tokens = response.prompt_eval_count or 0
            print(f"turn {turn}: {tokens} tokens, tools loaded = {len(tools)}")

        messages.append(response.message)

        if not response.message.tool_calls:
            print(f"\nANSWER: {response.message.content}\n")
            return messages

        for call in response.message.tool_calls:
            name = call.function.name
            args = call.function.arguments

            if name == "search_tools":
                found = search_tools(**args)
                if verbose:
                    print(f"  [search] {args} -> {found}")
                messages.append({
                    "role": "tool", "tool_name": "search_tools",
                    "content": json.dumps({"found": found}),
                })
                for tool_name in found:
                    fn = REGISTRY[tool_name]
                    if fn not in tools:
                        tools.append(fn)
            else:
                if verbose:
                    print(f"  [call] {name}({args})")
                result = REGISTRY[name](**args)
                messages.append({
                    "role": "tool", "tool_name": name,
                    "content": json.dumps(result),
                })

    print("stopped: hit max_turns without a final answer")
    return messages

In [72]:
run_with_deferred_loading("Give me the sales summary for the last 7 days.")

turn 1: 450 tokens, tools loaded = 1
  [search] {'query': 'sales summary report'} -> ['get_low_stock_items', 'create_support_ticket', 'get_sales_summary']
turn 2: 901 tokens, tools loaded = 4
  [call] get_sales_summary({'category': 'all', 'period': 'last 7 days'})
turn 3: 1058 tokens, tools loaded = 4

ANSWER: The sales summary for the last 7 days is as follows:

*   **Total Revenue:** ₹4,823,000
*   **Units Sold:** 214
*   **Average Order Value:** ₹22,537

**Top Products:**
1.  **SoundCore Buds** (58 units)
2.  **AeroBook 14** (31 units)



[{'role': 'system',
  'content': "You are an operations assistant with access to many tools. search_products is already loaded and available - use it immediately whenever you have a product name but need its product_id. All other tools are discovered on demand through search_tools.\n\nYou can call search_tools as many times as needed within a single task - once for each distinct capability you're missing, not just once per task.\n\nIf you're missing something a tool needs - an ID, a lookup value, a prior result, or any other piece of data - that almost always means you're missing a TOOL, not missing information from the user. Search for a tool that can produce it. Chain tools together: use the output of one tool as the input to the next.\n\nOnly ask the user a question when no reasonable tool could resolve it. Never ask the user for something a tool could look up for you."},
 {'role': 'user', 'content': 'Give me the sales summary for the last 7 days.'},
 Message(role='assistant', conte

## AGENTIC LOOP WITHOUT DEFERRED LOADING

In [73]:
# 1. Initialize the messages list outside the function with just the system prompt
messages = [
    {"role": "system", "content": SYSTEM_PROMPT}
]

def run(question, max_turns=10):
    # 2. Append the new user question to the existing external list
    messages.append({"role": "user", "content": question})

    for turn in range(1, max_turns + 1):
        response = ollama.chat(
            model=MODEL,
            messages=messages,
            tools=REAL_TOOLS,
            options={"num_ctx": 8000}
        )

        # the size of the ENTIRE context the model just read
        size = response.prompt_eval_count or 0
        print(f"turn {turn:>2}: {size:>6} tokens  {'#' * (size // 200)}")

        messages.append(response.message)

        if not response.message.tool_calls:
            print(f"\nANSWER: {response.message.content}\n")
            break

        for call in response.message.tool_calls:
            name = call.function.name
            args = call.function.arguments
            print(f"           -> {name}({args})")

            if name in REGISTRY:
                result = REGISTRY[name](**args)
            else:
                result = {"error": f"no tool named '{name}'"}

            # the line that grows the context - results are appended, never removed
            messages.append({
                "role": "tool",
                "tool_name": name,
                "content": json.dumps(result),
            })
    else:
        print("stopped: hit max_turns")

    print(f"messages in history: {len(messages)}")
    return messages

In [74]:
import time
start= time.time()
history = run("Give me the sales summary for the last 7 days.")
end= time.time()

print(f"Difference:- {end-start}")

turn  1:   2078 tokens  ##########
           -> get_sales_summary({'category': 'all', 'period': 'last 7 days'})
turn  2:   2231 tokens  ###########

ANSWER: The sales summary for the last 7 days is as follows:

*   **Total Revenue:** ₹4,823,000
*   **Units Sold:** 214
*   **Average Order Value:** ₹22,537
*   **Top Products:**
    *   **SoundCore Buds:** 58 units
    *   **AeroBook 14:** 31 units

messages in history: 5
Difference:- 4.286726474761963


## AGNETIC LOOP WITH DEFERRED LOADING

In [75]:
# Cell 11 - THE ONLY run_with_deferred_loading definition in this notebook.
# search_products is always loaded alongside SEARCH_TOOL - it's the "glue"
# tool almost everything else depends on, so it shouldn't need discovering.

def run_with_deferred_loading(question, max_turns=8, verbose=True):
    tools = [SEARCH_TOOL]
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]

    for turn in range(1, max_turns + 1):
        response = ollama.chat(
            model=MODEL,
            messages=messages,
            tools=tools,
            options={"num_ctx":8000},   # FIX #1 - avoid silent truncation
        )

        if verbose:
            tokens = response.prompt_eval_count or 0
            print(f"turn {turn}: {tokens} tokens, tools loaded = {len(tools)}")

        messages.append(response.message)

        if not response.message.tool_calls:
            print(f"\nANSWER: {response.message.content}\n")
            return messages

        for call in response.message.tool_calls:
            name = call.function.name
            args = call.function.arguments

            if name == "search_tools":
                found = search_tools(**args)
                if verbose:
                    print(f"  [search] {args} -> {found}")
                messages.append({
                    "role": "tool", "tool_name": "search_tools",
                    "content": json.dumps({"found": found}),
                })
                for tool_name in found:
                    fn = REGISTRY[tool_name]
                    if fn not in tools:
                        tools.append(fn)
            else:
                if verbose:
                    print(f"  [call] {name}({args})")
                result = REGISTRY[name](**args)
                messages.append({
                    "role": "tool", "tool_name": name,
                    "content": json.dumps(result),
                })

    print("stopped: hit max_turns without a final answer")
    return messages

In [76]:
import time

In [77]:
start= time.time()
history = run_with_deferred_loading("Give me the sales summary for the last 7 days.")
end= time.time()

print(f"Difference:- {end-start}")

turn 1: 450 tokens, tools loaded = 1
  [search] {'query': 'sales summary report'} -> ['get_low_stock_items', 'create_support_ticket', 'get_sales_summary']
turn 2: 901 tokens, tools loaded = 4
  [call] get_sales_summary({'category': 'all', 'period': 'last 7 days'})
turn 3: 1058 tokens, tools loaded = 4

ANSWER: The sales summary for the last 7 days is as follows:

*   **Total Revenue:** ₹4,823,000
*   **Units Sold:** 214
*   **Average Order Value:** ₹22,537
*   **Top Products:** 
    *   **SoundCore Buds:** 58 units
    *   **AeroBook 14:** 31 units

Difference:- 6.984571933746338


## OTHER EXAMPLES

In [78]:
import time

start= time.time()
response= run("hi how are you")
end= time.time()

print(f"Difference :- {end-start}")

turn  1:   2341 tokens  ###########

ANSWER: I'm doing well, thank you for asking! How can I help you with your operations or any customer queries today?

messages in history: 7
Difference :- 1.344874382019043


In [79]:
import time

start= time.time()
response= run_with_deferred_loading("hi how are you")
end= time.time()

print(f"Difference :- {end-start}")

turn 1: 442 tokens, tools loaded = 1

ANSWER: I'm doing well, thank you for asking! How can I help you today?

Difference :- 0.7933127880096436


In [80]:
import time

start= time.time()
response= run("What is price of AeroBook 14")
end= time.time()

print(f"Difference :- {end-start}")

turn  1:   2385 tokens  ###########
           -> search_products({'category': 'all', 'max_results': 5, 'query': 'AeroBook 14'})
turn  2:   2571 tokens  ############

ANSWER: The price of the AeroBook 14 is ₹94,990.

messages in history: 11
Difference :- 30.435424327850342


In [85]:
import time

start= time.time()
response= run_with_deferred_loading("What is price of AeroBook 14")
end= time.time()

print(f"Difference :- {end-start}")

turn 1: 447 tokens, tools loaded = 1
  [call] search_products({'query': 'AeroBook 14'})
turn 2: 621 tokens, tools loaded = 1

ANSWER: The price of the AeroBook 14 is ₹94,990.

Difference :- 1.9509100914001465


## Embedding-based search (hybrid: keyword + meaning)

In [86]:
# Cell 5 - embedding helpers

def get_embedding(text, task_type="search_document"):
    response = ollama.embed(model=EMBED_MODEL, input=f"{task_type}: {text}")
    return response["embeddings"][0]


def cosine_similarity(a, b):
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))


def clean_docstring(doc):
    if not doc:
        return ""
    return re.split(r"\n\s*(Args:|Returns:)", doc)[0].strip()

In [87]:
# Cell 6 - precompute embeddings ONCE, not inside search_tools()

TOOL_EMBEDDINGS = {}
for f in REAL_TOOLS:
    text = f.__name__ + " " + clean_docstring(f.__doc__) # concatinating name and docstring
    TOOL_EMBEDDINGS[f.__name__] = get_embedding(text, task_type="search_document")

print(f"Precomputed embeddings for {len(TOOL_EMBEDDINGS)} tools")

Precomputed embeddings for 15 tools


In [88]:
def search_tools(query, max_results=3):
    query_emb = get_embedding(query, task_type="search_query")
    query_words = set(query.lower().split())

    scored = []
    for f in REAL_TOOLS:
        name = f.__name__
        text = (name + " " + clean_docstring(f.__doc__)).lower()
        tool_words = set(text.split())

        keyword_score = len(query_words & tool_words) / max(len(query_words), 1)
        embedding_score = cosine_similarity(query_emb, TOOL_EMBEDDINGS[name])
        combined = (0.6 * keyword_score) + (0.4 * embedding_score)
        scored.append((combined, name))

    scored.sort(reverse=True)
    return [name for score, name in scored[:max_results]]


## The search tool schema, system prompt, and agent loop

In [89]:
# Cell 9 - THE ONLY SEARCH_TOOL definition in this notebook

SEARCH_TOOL = {
    "type": "function",
    "function": {
        "name": "search_tools",
        "description": (
            "Search for tools that are not yet loaded into this conversation, "
            "by meaning as well as keywords. Use this the moment you realize "
            "you need a capability you don't currently have.\n\n"
            "Returns matching tool names, ranked by relevance. Once a tool is "
            "returned here, it becomes directly callable on your next turn - "
            "you do not need to search for it again later.\n\n"
            "If the top result doesn't look right, try rephrasing your query "
            "rather than giving up."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": (
                        "Describe the capability you need in a natural phrase - "
                        "full context helps more than a few bare keywords."
                    ),
                },
                "max_results": {
                    "type": "integer",
                    "description": "Max tools to return. Defaults to 3.",
                },
            },
            "required": ["query"],
        },
    },
}

In [91]:
# Cell 11 - THE ONLY run_with_deferred_loading definition in this notebook.
# search_products is always loaded alongside SEARCH_TOOL - it's the "glue"
# tool almost everything else depends on, so it shouldn't need discovering.

def run_with_deferred_loading(question, max_turns=8, verbose=True):
    tools = [SEARCH_TOOL]
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]

    for turn in range(1, max_turns + 1):
        response = ollama.chat(
            model=MODEL,
            messages=messages,
            tools=tools,
            options={"num_ctx":8000},   # FIX #1 - avoid silent truncation
        )

        if verbose:
            tokens = response.prompt_eval_count or 0
            print(f"turn {turn}: {tokens} tokens, tools loaded = {len(tools)}")

        messages.append(response.message)

        if not response.message.tool_calls:
            print(f"\nANSWER: {response.message.content}\n")
            return messages

        for call in response.message.tool_calls:
            name = call.function.name
            args = call.function.arguments

            if name == "search_tools":
                found = search_tools(**args)
                if verbose:
                    print(f"  [search] {args} -> {found}")
                messages.append({
                    "role": "tool", "tool_name": "search_tools",
                    "content": json.dumps({"found": found}),
                })
                for tool_name in found:
                    fn = REGISTRY[tool_name]
                    if fn not in tools:
                        tools.append(fn)
            else:
                if verbose:
                    print(f"  [call] {name}({args})")
                result = REGISTRY[name](**args)
                messages.append({
                    "role": "tool", "tool_name": name,
                    "content": json.dumps(result),
                })

    print("stopped: hit max_turns without a final answer")
    return messages

In [92]:
start= time.time()
run_with_deferred_loading("What is the current stock level for the AeroBook 14 laptop?")
end= time.time()
print(f"Difference:- {end-start}")

turn 1: 400 tokens, tools loaded = 1
  [call] search_products({'query': 'AeroBook 14'})
turn 2: 574 tokens, tools loaded = 1
  [search] {'query': 'get current stock level for a product id'} -> ['get_inventory_level', 'get_low_stock_items', 'get_product_details']
turn 3: 1016 tokens, tools loaded = 4
  [call] get_inventory_level({'product_id': 'P-1001', 'warehouse': 'all'})
turn 4: 1130 tokens, tools loaded = 4

ANSWER: The current stock level for the AeroBook 14 (P-1001) is 47 total units. 

Of these, 8 units are reserved for existing orders, leaving 39 units available for sale. The stock is distributed as follows:
- **Mumbai:** 30 units
- **Delhi:** 12 units
- **Bangalore:** 5 units

Difference:- 35.03249001502991


In [93]:
start= time.time()
run("What is the current stock level for the AeroBook 14 laptop?")
end= time.time()

print(f"Difference:- {end-start}")

turn  1:   2619 tokens  #############
           -> get_inventory_level({'product_id': 'P-1001', 'warehouse': 'all'})
turn  2:   2729 tokens  #############

ANSWER: The current stock level for the AeroBook 14 is 47 units in total. However, 8 units are reserved for existing orders, leaving 39 units available for sale.

Here is the breakdown by warehouse:
*   **Mumbai:** 30 units
*   **Delhi:** 12 units
*   **Bangalore:** 5 units

messages in history: 15
Difference:- 26.733196020126343
